# LLM Load Balancing — VPA only

Same load-shift experiment as `load_balancing.ipynb`, but evaluated against three local LLMs (**qwen2.5:7b, mistral, llama3:8b**) instead of trained RL agents.

The load distribution across services is **flipped halfway through the run**:
- First half: most load on service 1
- Second half: most load on service N

A good algorithm should detect the shift and **move CPU budget** from the previously-hot service to the newly-hot one. Replica scaling is disabled — this notebook tests **vertical scaling quality** in isolation.

**Configuration:** `n_services = 2` by default, but bump `n_services` and the `first_loads`/`second_loads` lists to scale up to 5 once you have more Pis.


In [ ]:
import os
import pickle
import subprocess
import sys
import time
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, '../src')

from llm_agent import LLMAgent, load_llm_config
from llm_providers import OllamaProvider
from envs import JointContinuousElasticityEnv, set_available_resource, set_other_priorities, set_other_utilization
from pod_controller import set_container_cpu_values, get_loadbalancer_external_port
from spam_cluster import get_response_times
from utils import init_nodes

# Run from project root (same convention as the existing notebook)
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('cwd:', os.getcwd())

In [ ]:
# --- Experiment config ---

# Cluster: edit these to scale up when you add more Pis (up to 5 services).
n_services = 2
# Load distribution across services. Must have length == n_services and sum to ~1.0.
# At n_services=2 we make service 1 the initial hot one, then flip.
first_loads  = [0.7, 0.3]
second_loads = [0.3, 0.7]

# Models to test. Tags are the Ollama model names exactly as `ollama list` shows them.
models = [
    ('qwen2.5:7b',     'qwen2.5:7b'),
    ('mistral:latest', 'mistral'),
    ('llama3:8b',      'llama3:8b'),
]

# Load generator settings (same shape as load_balancing.ipynb)
rps = 60              # total requests/sec across all services
interval = 1000       # ms between request bursts in spam_cluster.py
recordings = 30       # steps per iteration (load flips at recordings//2)
time_step = 1.0       # seconds per step (matches benchmark_runner's action_interval)
num_iterations = 3    # iterations to average over per model
initial_container_cpu = 50   # millicores baseline before run starts
max_cpu = 1000        # CPU budget passed to LLMAgent (sets prompt's max_cpu rule)
USERS = 5             # concurrent probe users per response-time sample

assert len(first_loads) == n_services and len(second_loads) == n_services, \
    'first_loads / second_loads must have length n_services'
assert abs(sum(first_loads) - 1.0) < 0.01 and abs(sum(second_loads) - 1.0) < 0.01, \
    'load lists must sum to 1.0'
print(f'{n_services} services, {num_iterations} iters/model, {recordings} steps/iter')
print('first_loads:', first_loads)
print('second_loads:', second_loads)

In [ ]:
# --- Helpers ---

def make_agent(model_tag, service_idx):
    """Spin up a fresh LLMAgent + env pair for one service."""
    llm_cfg = load_llm_config()
    oll_cfg = llm_cfg.get('ollama', {})
    provider = OllamaProvider(
        model=model_tag,
        base_url=oll_cfg.get('base_url', 'http://localhost:11434'),
        max_tokens=oll_cfg.get('max_tokens', 512),
    )
    deployment_name = 'localization-api'
    env = JointContinuousElasticityEnv(service_idx, pod_name=None)
    env.MAX_CPU_LIMIT = max_cpu
    agent = LLMAgent(
        provider,
        pod_name=f'{deployment_name}{service_idx}',
        deployment_name=deployment_name,
        history_window=5,
        inference_mode='parser',
        max_cpu=max_cpu,
    )
    return env, agent


def vpa_only(action):
    """Force the HPA component to zero so this experiment tests VPA only."""
    a = np.asarray(action, dtype=np.float32).flatten()
    if a.size >= 2:
        a[1] = 0.0
    return a


# Cluster handle (used for direct container readings every step)
url = f'http://localhost:{get_loadbalancer_external_port(service_name="ingress-nginx-controller")}'
nodes = init_nodes(debug=True, custom_label='app=localization-api')
print('url =', url)
print('nodes:', [getattr(n, 'name', n) for n in nodes])

## VPA-only prompt override

The default `LLMAgent` prompt offers both `scale_cpu` and `scale_replicas`. For this load-balancing experiment we want **vertical scaling only**. The cell below replaces the parser-mode prompt with a VPA-focused version that:

- Removes the `scale_replicas` tool from the allowed call list
- Explicitly tells the model not to scale replicas
- Adds load-balancing-specific guidance about shifting CPU between hot and cold pods

Edit this string freely — it's monkey-patched onto `llm_agent` at module level, so every `LLMAgent` created afterwards picks it up. The code-level `vpa_only()` safeguard from cell 3 stays in place as a backstop in case the model ignores the prompt.

In [ ]:
import llm_agent as _la

# === Edit this prompt to experiment with phrasing ===
VPA_ONLY_PROMPT = """You are an autoscaling agent for a Kubernetes cluster running microservices on edge devices (Raspberry Pi).
Your job is to manage CPU resources efficiently by adjusting per-pod CPU limits.

In this experiment you have ONE tool: scale_cpu. Replica scaling is disabled and will be ignored if you call it.

Context:
- The system runs multiple service pods sharing a fixed CPU budget.
- One service may be receiving most of the load while another is mostly idle.
- Your job is to detect that and redistribute CPU: take from idle pods, give to busy pods.

Goals:
- Keep CPU utilization between 30-60% per pod (target range)
- Minimize response latency (keep under 250ms)
- Do not waste budget: if your pod is at <20% utilization, lower its CPU limit so other pods can use that headroom.
- If your pod is saturated (utilization >70% or response time >250ms) and budget is available, raise the limit aggressively.

Rules:
- CPU limits range from 50m to {max_cpu}m per pod.
- Available shared resources are limited; increasing one pod reduces what's available for others.
- Do NOT call scale_replicas in this experiment.

After your reasoning, end your response with EXACTLY ONE of:
  scale_cpu(<delta_millicores>)   integer delta, e.g. scale_cpu(50) or scale_cpu(-100)
  no_action()
"""

# Monkey-patch the parser-mode prompt. Every LLMAgent(inference_mode='parser') created
# AFTER this cell runs will use VPA_ONLY_PROMPT instead of the default SYSTEM_PROMPT_PARSER.
_la.SYSTEM_PROMPT_PARSER = VPA_ONLY_PROMPT
print('VPA-only prompt installed.')
print('Length:', len(VPA_ONLY_PROMPT), 'chars')

In [ ]:
# --- Main experiment loop ---
# Records per (model, iteration): a list of per-step (cpu_limit, cpu_usage) tuples per service,
# and a list of per-step mean response times per service.

crec_alg = defaultdict(list)   # model_tag -> [iteration][step][service] = (cpu_limit, cpu_usage, cpu_pct)
rts_alg  = defaultdict(list)   # model_tag -> [iteration][service] = [rt_per_step]

for model_tag, display_name in models:
    print(f'\n========== {display_name} ({model_tag}) ==========')
    for iteration in range(num_iterations):
        print(f'  iter {iteration}')

        # Reset cluster: high cpu for cleanup, then back to baseline.
        set_container_cpu_values(cpus=1000, n=n_services)
        time.sleep(2)
        set_container_cpu_values(cpus=initial_container_cpu, n=n_services)
        time.sleep(5)

        # Fresh agents per iteration so state/history is clean
        envs_and_agents = [make_agent(model_tag, i) for i in range(1, n_services + 1)]
        envs   = [ea[0] for ea in envs_and_agents]
        agents = [ea[1] for ea in envs_and_agents]
        other_envs = [[e for e in envs if e is not envs[i]] for i in range(len(envs))]
        states = [np.asarray(e.reset()).flatten() for e in envs]
        set_available_resource(envs, max_cpu)

        # Launch first load distribution
        def spawn_load(loads):
            cmds = []
            for i, frac in enumerate(loads):
                cmds.append(['python', 'src/spam_cluster.py',
                             '--users', str(int(rps * frac)),
                             '--interval', str(interval),
                             '--service', str(i + 1)])
            return [subprocess.Popen(c) for c in cmds]

        subs = spawn_load(first_loads)

        container_recordings = []
        rts_per_service = {i: [] for i in range(1, n_services + 1)}

        try:
            for step in range(recordings):
                # Mid-run load flip
                if step == recordings // 2:
                    for p in subs:
                        p.terminate(); p.wait()
                    subs = spawn_load(second_loads)
                    print(f'    step {step}: load flipped to {second_loads}')

                t0 = time.time()

                # 1. Sample response time per service
                rts = []
                for i, env in enumerate(envs):
                    sample = get_response_times(USERS, f'{url}/api{i+1}/predict')
                    valid = [r for r in sample if r is not None]
                    mean_rt = float(np.mean(valid)) if valid else 2.0
                    rts.append(mean_rt)
                    rts_per_service[i + 1].append(mean_rt)

                # 2. Each agent decides (VPA-only) and the env applies the action
                for i, agent in enumerate(agents):
                    set_other_utilization(envs[i], other_envs[i])
                    set_other_priorities(envs[i], other_envs[i])
                    if hasattr(agent, 'observe_response_time'):
                        agent.observe_response_time(rts[i])
                    action = agent.get_action(states[i])
                    action = vpa_only(action)
                    state, reward, done, _ = envs[i].step(action, 2)
                    set_available_resource(envs, max_cpu)
                    states[i] = np.asarray(state).flatten()

                # 3. Record per-service container stats
                step_rec = []
                for i in range(1, n_services + 1):
                    cpu_limit_total, cpu_used_total, cpu_pct_total, count = 0, 0, 0, 0
                    for node in nodes:
                        node.update_containers(debug=False, custom_label='app=localization-api',
                                               reset_containers=True)
                        for container_id, (pod_name, _, _) in list(node.get_containers().items()):
                            if f'localization-api{i}' in pod_name or f'api{i}' in pod_name:
                                (cpu_limit, cpu, cpu_percentage), _, _, _ = node.get_container_usage(container_id)
                                cpu_limit_total += cpu_limit
                                cpu_used_total  += cpu
                                cpu_pct_total   += cpu_percentage
                                count += 1
                    if count == 0:
                        step_rec.append((float('nan'), float('nan'), float('nan')))
                    else:
                        step_rec.append((cpu_limit_total, cpu_used_total, cpu_pct_total / count))
                container_recordings.append(step_rec)

                elapsed = time.time() - t0
                time.sleep(max(0, time_step - elapsed))

        finally:
            for p in subs:
                p.terminate(); p.wait()

        crec_alg[model_tag].append(container_recordings)
        rts_alg[model_tag].append(rts_per_service)

print('\nDone.')

In [ ]:
# --- Aggregate across iterations ---
# For each model:
#   mean_crec[model][step][service]   = (mean_cpu_limit, mean_cpu_used, mean_cpu_pct)
#   mean_rts[model][service][step]    = mean_rt

mean_crec = {}
mean_rts  = {}

for model_tag, _ in models:
    iters_crec = crec_alg.get(model_tag, [])
    iters_rts  = rts_alg.get(model_tag, [])
    if not iters_crec:
        continue

    # Containers
    per_step = []
    for step in range(recordings):
        per_service = []
        for service_idx in range(n_services):
            vals = [it[step][service_idx] for it in iters_crec
                    if len(it) > step and not any(np.isnan(it[step][service_idx]))]
            if vals:
                cl = np.mean([v[0] for v in vals])
                cu = np.mean([v[1] for v in vals])
                cp = np.mean([v[2] for v in vals])
            else:
                cl, cu, cp = float('nan'), float('nan'), float('nan')
            per_service.append((cl, cu, cp))
        per_step.append(per_service)
    mean_crec[model_tag] = per_step

    # Response times
    rts_by_service = {sid: [] for sid in range(1, n_services + 1)}
    for sid in range(1, n_services + 1):
        for step in range(recordings):
            vals = []
            for it in iters_rts:
                if len(it[sid]) > step:
                    vals.append(it[sid][step])
            rts_by_service[sid].append(float(np.mean(vals)) if vals else float('nan'))
    mean_rts[model_tag] = rts_by_service

print('Aggregated:', list(mean_crec.keys()))

In [ ]:
# Optional: save aggregated results so re-plotting doesn't require re-running the experiment
os.makedirs('results/llm_load_balancing', exist_ok=True)
pickle.dump(mean_crec, open('results/llm_load_balancing/mean_crec.p', 'wb'))
pickle.dump(mean_rts,  open('results/llm_load_balancing/mean_rts.p',  'wb'))
print('saved to results/llm_load_balancing/')

## Per-model overview: response time and CPU allocation

Two-column grid, one row per model. Left column = mean response time per service over time; right column = mean CPU allocation per service over time. Dashed vertical line marks the load flip.

In [ ]:
n_models = len(mean_crec)
fig, axes = plt.subplots(n_models, 2, figsize=(15, 4 * n_models), squeeze=False)

colors = plt.cm.tab10.colors

for row, (model_tag, display_name) in enumerate(models):
    if model_tag not in mean_crec:
        continue
    crec = mean_crec[model_tag]
    rts  = mean_rts[model_tag]
    time_axis = list(range(len(crec)))
    halfway = len(time_axis) // 2

    ax1 = axes[row, 0]
    for sid in range(1, n_services + 1):
        ax1.plot(time_axis, [v * 1000 for v in rts[sid]],
                 label=f'service {sid}', color=colors[(sid - 1) % len(colors)], linewidth=1.6)
    ax1.axvline(halfway, color='gray', linestyle='--', alpha=0.7)
    ax1.axhline(250, color='gray', linestyle=':', alpha=0.5)
    ax1.text(halfway, ax1.get_ylim()[1] * 0.95, 'load flip', color='gray', fontsize=9)
    ax1.set_xlabel('Step (s)')
    ax1.set_ylabel('Mean RT (ms)')
    ax1.set_title(f'{display_name} — Response time per service')
    ax1.legend(loc='upper left', fontsize=9)
    ax1.grid(True, alpha=0.3)

    ax2 = axes[row, 1]
    for sid in range(n_services):
        cl = [step[sid][0] for step in crec]
        cu = [step[sid][1] for step in crec]
        ax2.plot(time_axis, cl, label=f'service {sid+1} limit',
                 color=colors[sid % len(colors)], linewidth=1.6, linestyle='--')
        ax2.plot(time_axis, cu, label=f'service {sid+1} used',
                 color=colors[sid % len(colors)], linewidth=1.4)
    ax2.axvline(halfway, color='gray', linestyle='--', alpha=0.7)
    ax2.set_xlabel('Step (s)')
    ax2.set_ylabel('CPU (millicores)')
    ax2.set_title(f'{display_name} — CPU allocation (dashed) and usage (solid)')
    ax2.legend(loc='upper left', fontsize=8)
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Comparison view: each service's RT across all models

Per service, one line per model — shows which LLM reacts fastest to the load shift on that service.

In [ ]:
fig, axes = plt.subplots(n_services, 1, figsize=(13, 4 * n_services), squeeze=False)

for sid in range(1, n_services + 1):
    ax = axes[sid - 1, 0]
    halfway = recordings // 2
    ax.axvspan(0, halfway, color='#e8f4f8', alpha=0.6, zorder=0)
    ax.axvspan(halfway, recordings, color='#fde8e8', alpha=0.6, zorder=0)
    ax.text(halfway / 2, 0.95, f'load = {first_loads[sid-1]}',
            transform=ax.get_xaxis_transform(), ha='center', va='top', color='gray')
    ax.text(halfway + (recordings - halfway) / 2, 0.95, f'load = {second_loads[sid-1]}',
            transform=ax.get_xaxis_transform(), ha='center', va='top', color='gray')

    for idx, (model_tag, display_name) in enumerate(models):
        if model_tag not in mean_rts:
            continue
        ax.plot(range(recordings), [v * 1000 for v in mean_rts[model_tag][sid]],
                label=display_name, color=colors[idx % len(colors)], linewidth=1.7)

    ax.axhline(250, color='gray', linestyle='--', linewidth=1, alpha=0.6)
    ax.set_xlabel('Step (s)')
    ax.set_ylabel('Mean RT (ms)')
    ax.set_title(f'Service {sid} response time across models')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, recordings)

plt.tight_layout()
plt.show()

## Summary statistics

Mean RT per model, before vs. after the flip. A model that successfully rebalances should have similar RT in both halves; a model that doesn't will degrade on the newly-hot service after the flip.

In [ ]:
rows = []
for model_tag, display_name in models:
    if model_tag not in mean_rts:
        continue
    halfway = recordings // 2
    for sid in range(1, n_services + 1):
        rts_arr = np.array(mean_rts[model_tag][sid])
        before = float(np.nanmean(rts_arr[:halfway])) * 1000
        after  = float(np.nanmean(rts_arr[halfway:])) * 1000
        rows.append({
            'model': display_name,
            'service': sid,
            'mean_rt_before_ms': round(before, 1),
            'mean_rt_after_ms':  round(after, 1),
            'change_ms':         round(after - before, 1),
        })

import pandas as pd
pd.DataFrame(rows)